# minillm 클라우드 학습 (Kaggle / Colab 무료 GPU)

로컬(CPU)에서는 코드 검증만 하고, **실제 학습은 이 노트북에서 무료 GPU로** 돌린다.

**Kaggle 사용법** (권장 — 주 30시간 무료, 세션 12h):
1. kaggle.com → Code → New Notebook
2. 우측 Settings → Accelerator → **GPU T4 x2** (또는 P100)
3. 이 노트북 셀을 위에서부터 실행
4. 세션이 끊기면 마지막 `--resume` 셀만 다시 실행하면 이어서 학습된다.

> `GITHUB_REPO`를 본인 저장소 주소로 바꿔라. 체크포인트는 세션 종료 시
> 사라지므로, 중간중간 `/kaggle/working`의 `checkpoints/`를 **Output으로 저장**하거나
> 마지막 셀로 다운로드해 로컬에 보관한다.

In [ ]:
GITHUB_REPO = 'https://github.com/<YOUR_ID>/minillm.git'  # <-- 본인 저장소로 변경
!git clone $GITHUB_REPO
%cd minillm
!pip install -q regex datasets tqdm
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. 데이터 준비 (최초 1회)
나무위키+위키 **혼합** 다운로드 → 토크나이저 학습(코퍼스 전역 랜덤샘플) → 토큰화(.bin).
처음엔 `--max-docs`로 작게 시험하고, 잘 되면 인자를 빼고 전체로 돌린다.
`DATASOURCES.txt`에 라이선스가 기록된다(나무위키는 cc-by-nc-sa, 비상업 개인용).

In [ ]:
# 전체를 받으려면 --max-docs 를 지운다. mix = 위키:나무 1:1 (--mix-ratio로 조절)
# vocab 16392 = 기본 특수 토큰 4개 + 마음 유사 기제용 예약 8개 (<|pause|> 등)
!python -m data.download --source mix --max-docs 50000
!python -m tokenizer.train_tokenizer --input data/raw/corpus.txt --vocab-size 16392 --sample-mb 200
!python -m data.pack --input data/raw/corpus.txt --tokenizer tokenizer/tokenizer.json

## 2. 사전학습 (base — 권장 경로)
`full`은 **base**다: loop off + `torch.compile` on + 토큰예산 1B(~4000스텝). 제일 비싼
사전학습을 빠르게 끝내고, 마음 기제는 SFT에서 붙인다. T4는 하드웨어 bf16이 없어
**fp16이 자동 선택**되고(로그 `AMP dtype: float16`), **Muon** 옵티마이저로 목표 loss까지
스텝 수를 줄인다.

**쿼터 예산**: 시작 로그의 `토큰 예산 1.0B -> N steps`와 첫 100스텝 `ms/step`으로
예상 시간 = N × ms/step 을 추정해 12h 세션에 맞춘다. 끊기면 `--resume`.
loop를 사전학습부터 넣고 싶으면 `--preset full-loop`.

In [ ]:
!python -m train.pretrain --preset full --optimizer muon            # 처음 시작
# !python -m train.pretrain --preset full --optimizer muon --resume  # 세션 재개 시 이 줄
# !python -m train.pretrain --preset full --target-tokens 500000000  # 예산 스윕(0.5B)

### (선택) 반복 횟수별 성능 — full-loop로 학습한 경우만
base(`full`)는 loop off라 이 비교는 `full-loop` 체크포인트에서만 의미가 있다.
같은 가중치로 n_loop 1/2/3의 val loss를 비교한다.

In [ ]:
!python -m tools.eval_loop --ckpt checkpoints/ckpt_best.pt --n-loops 1,2,3

## 3. 대화 파인튜닝 (SFT)
사전학습 best 체크포인트를 이어받아 대화 형식을 가르친다. 마음 기제는 여기서
단계별로 켠다 — 같은 base에서 갈라 학습해 val loss를 비교하는 것이 곧 실험이다:
- `--n-pause 4` : 답변 앞 <|pause|> — 가장 싼 '생각 시간' (baseline)
- `--mood-dim 64` : 턴 지속 기분 벡터 · `--latent 2` : Coconut 잠재 사고
- `--workspace-slots 4` : GWT 지속 작업공간 · `--attn-schema` : AST 주의 도식
- `--feedback` / `--conf` : 역피드백 / 확신도 헤드(메타인지)

In [ ]:
# 기본 + pause 4개
!python -m data.prepare_sft --tokenizer tokenizer/tokenizer.json --n-pause 4
!python -m train.sft --init checkpoints/ckpt_best.pt --data data/bin/sft.npz --out checkpoints/sft_pause4.pt --n-pause 4

# 새 마음 기제 (base에 조합):
# !python -m train.sft --init checkpoints/ckpt_best.pt --data data/bin/sft.npz --out checkpoints/sft_ws.pt --workspace-slots 4
# !python -m train.sft --init checkpoints/ckpt_best.pt --data data/bin/sft.npz --out checkpoints/sft_schema.pt --attn-schema --conf

# 잠재 vs pause 정면 비교(같은 k):
# !python -m data.prepare_sft --tokenizer tokenizer/tokenizer.json --out data/bin/sft_plain.npz
# !python -m train.sft --init checkpoints/ckpt_best.pt --data data/bin/sft_plain.npz --out checkpoints/sft_latent2.pt --latent 2

# --- 검증 하네스 (D1~D6) ---
# !python -m tools.eval_conf --ckpt checkpoints/sft_schema.pt --data data/bin/sft.npz --save
# !python -m tools.eval_schema --ckpt checkpoints/sft_schema.pt --data data/bin/sft.npz --save
# !python -m tools.eval_state --ckpt checkpoints/sft_ws.pt --save
# !python -m tools.eval_thinking --data data/bin/sft.npz --ckpts k0=checkpoints/sft_pause4.pt latent=checkpoints/sft_latent2.pt

## 4. 체크포인트 저장
`checkpoints/sft.pt`와 `tokenizer/tokenizer.json`을 로컬로 내려받아
`python chat.py --ckpt checkpoints/sft.pt`로 대화한다. (Kaggle은 오른쪽 Output 패널에서 다운로드)

In [ ]:
import shutil, os, glob
os.makedirs('/kaggle/working/out', exist_ok=True)
for f in glob.glob('checkpoints/*.pt') + ['tokenizer/tokenizer.json']:
    if os.path.exists(f):
        shutil.copy(f, '/kaggle/working/out/')
print('저장 완료 — Output 패널에서 다운로드하세요')